In [ ]:
from pathlib import Path
import os
import sys
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)

In [ ]:
from eintelligence.data_prep.aoi import square_aoi
from eintelligence.data_prep.temporal_pairing import build_landcover_multisensor_manifest
from eintelligence.data_prep.manifest_utils import merge_record_manifests
from orchestrator.workflow_manager_landcover import (
    LandCoverWorkflowMS,
    TilingConfigLandCover,
    TrainingConfigLandCover,
)


In [ ]:
PIPELINE_STAGE = "ingest"
# "ingest", "regional_only", "pooled_only", "infer_only"

CASE_NAME = "landcover"
SENSOR_MODE = "s1s2"

tiling_cfg = TilingConfigLandCover(
    bands_s2=("B02", "B03", "B04", "B08"),
    bands_s1=("vv", "vh"),
    tile_size=256,
    stride=256,
    max_cloud=50,
    sensor_mode=SENSOR_MODE,
)

train_cfg = TrainingConfigLandCover(
    batch_size=4,
    num_epochs=100,
    lr=1e-4,
    amp=False,   # keep false for now since you just debugged NaNs
)

wf = LandCoverWorkflowMS(project_root, tiling_cfg, train_cfg)


In [ ]:
# %%
# Define multiple AOIs / date windows.
# Keep them sequential so downloads and tiling happen one run at a time.

RUN_SPECS = [
    {
        "region_name": "munich_lc_2023_summer",
        "aoi_geojson": square_aoi(48.1351, 11.5820),
        "start": "2023-06-01",
        "end": "2023-08-01",
        "aoi_id": "munich_core",
        "job_id": "munich_lc_2023_summer",
    },
    # Example:
    # {
    #     "region_name": "novo_progresso_lc_2023_dry",
    #     "aoi_geojson": square_aoi(-7.754, -55.513),
    #     "start": "2023-06-01",
    #     "end": "2023-09-01",
    #     "aoi_id": "novo_progresso",
    #     "job_id": "novo_progresso_lc_2023_dry",
    # },
]


In [ ]:
corpus_dir = Path(project_root) / "data" / "corpus"
corpus_dir.mkdir(parents=True, exist_ok=True)

pooled_manifest = corpus_dir / "landcover_manifest_multisensor.json"
splits_path = corpus_dir / "landcover_splits.json"

models_dir = Path(project_root) / "models"
models_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = models_dir / f"{CASE_NAME}_{SENSOR_MODE}.pt"
out_dir = corpus_dir / f"pred_{CASE_NAME}_{SENSOR_MODE}"

In [ ]:
def regional_manifest_path(project_root: str, region_name: str) -> Path:
    return Path(project_root) / "data" / region_name / "S2" / "landcover_manifest_multisensor.json"

In [ ]:
regional_manifest_paths = []
regional_namespaces = []

if PIPELINE_STAGE == "ingest":
    for spec in RUN_SPECS:
        region_name = spec["region_name"]
        print(f"\n=== INGEST: {region_name} ===")

        s1_coll, s2_coll, registry_path = wf.ingest_region(
            aoi_geojson=spec["aoi_geojson"],
            start=spec["start"],
            end=spec["end"],
            region_name=region_name,
            aoi_id=spec.get("aoi_id"),
            job_id=spec.get("job_id"),
        )

        landcover_manifest = build_landcover_multisensor_manifest(
            s2_collection_manifest_path=s2_coll,
            s1_collection_manifest_path=s1_coll,
            iou_min=0.8,
            worldcover_version="v200",
            worldcover_year="2021",
        )

        regional_manifest_paths.append(Path(landcover_manifest))
        regional_namespaces.append(region_name)
        print(f"[OK] built regional manifest: {landcover_manifest}")

elif PIPELINE_STAGE == "regional_only":
    for spec in RUN_SPECS:
        region_name = spec["region_name"]
        manifest_path = regional_manifest_path(project_root, region_name)

        if not manifest_path.is_file():
            raise RuntimeError(
                f"Expected regional manifest not found for {region_name}: {manifest_path}"
            )

        regional_manifest_paths.append(manifest_path)
        regional_namespaces.append(region_name)
        print(f"[OK] using existing regional manifest: {manifest_path}")

elif PIPELINE_STAGE in ("pooled_only", "infer_only"):
    print(f"[OK] skipping regional stage: {PIPELINE_STAGE}")

else:
    raise ValueError(f"Unsupported PIPELINE_STAGE: {PIPELINE_STAGE}")

In [ ]:
# Step 2: merge all regional manifests into one pooled manifest

if PIPELINE_STAGE in ("ingest", "regional_only"):
    pooled_manifest = merge_record_manifests(
        manifest_paths=regional_manifest_paths,
        out_path=pooled_manifest,
        record_key="tiles",
        task_name="landcover_multisensor",
        namespaces=regional_namespaces,
        fields_to_prefix=("group_id", "tile_id", "scene_id"),
        set_default_aoi_id=True,
        deduplicate_on="tile_id",
        sort_by=("aoi_id", "group_id", "scene_id", "datetime", "row", "col", "tile_id"),
    )
    print(f"[OK] pooled manifest: {pooled_manifest}")

elif PIPELINE_STAGE in ("pooled_only", "infer_only"):
    if not pooled_manifest.is_file():
        raise RuntimeError(f"Pooled manifest not found: {pooled_manifest}")
    print(f"[OK] using existing pooled manifest: {pooled_manifest}")

In [ ]:
if PIPELINE_STAGE == "infer_only":
    if not ckpt_path.is_file():
        raise RuntimeError(f"Checkpoint not found for infer_only: {ckpt_path}")

    wf.run(
        landcover_manifest=pooled_manifest,
        splits_path=splits_path,
        ckpt_path=ckpt_path,
        out_dir=out_dir,
        mode="infer",
        retrain=False,
        infer_split_name="test",
        stitch_scenes=False,
        max_tiles=None,
    )

else:
    wf.run(
        landcover_manifest=pooled_manifest,
        splits_path=splits_path,
        ckpt_path=ckpt_path,
        out_dir=out_dir,
        mode="train_and_infer",
        retrain=True,
        infer_split_name="test",
        stitch_scenes=False,
        max_tiles=None,
    )
